# Задание 2. Объединение нескольких файлов в один


In [1]:
import os


def create_source_files():
    """Создаёт три исходных файла с произвольным текстом."""
    contents = {
        "part1.txt": [
            "Первая строка первого файла",
            "Вторая строка первого файла",
            "Третья строка первого файла",
        ],
        "part2.txt": [
            "Первая строка второго файла",
            "Вторая строка второго файла",
            "Третья строка второго файла",
            "Четвёртая строка второго файла",
        ],
        "part3.txt": [
            "Первая строка третьего файла",
            "Вторая строка третьего файла",
            "Третья строка третьего файла",
        ],
    }
    for name, lines in contents.items():
        with open(name, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    return list(contents.keys())


def merge_files(names, result_name):
    """
    Объединяет содержимое файлов из списка names в файл result_name.
    Возвращает общее количество перенесённых строк (без служебных заголовков).
    """
    total_lines = 0
    toc = []          # оглавление: (имя_файла, строка_начала)
    current_line = 0  # текущая строка в итоговом файле

    with open(result_name, "w", encoding="utf-8") as out:

        for name in names:
            # Заголовок ===== part1.txt =====
            header = f"===== {name} =====\n"
            out.write(header)
            current_line += 1

            # Запоминаем, с какой строки начинается содержимое файла
            toc.append((name, current_line + 1))

            # Открываем и читаем исходный файл по очереди в цикле
            with open(name, "r", encoding="utf-8") as src:
                for line in src:
                    out.write(line)
                    current_line += 1
                    total_lines += 1

        # ==== Оглавление в конец файла ====
        out.write("\n===== ОГЛАВЛЕНИЕ =====\n")
        for name, start in toc:
            out.write(f"{name} — начиная со строки {start}\n")

    return total_lines


def main():
    # 1. Создаём исходные файлы
    names = create_source_files()
    print("Созданы исходные файлы:", ", ".join(names))

    # 2. Объединяем
    result_name = "merged.txt"
    count = merge_files(names, result_name)

    # 3. Выводим итоговый файл на экран
    print(f"\n===== Содержимое файла {result_name} =====")
    with open(result_name, "r", encoding="utf-8") as f:
        print(f.read(), end="")

    # 4. Размер в байтах
    size = os.path.getsize(result_name)
    print(f"\nОбщее количество перенесённых строк: {count}")
    print(f"Размер итогового файла: {size} байт")


if __name__ == "__main__":
    main()

Созданы исходные файлы: part1.txt, part2.txt, part3.txt

===== Содержимое файла merged.txt =====
===== part1.txt =====
Первая строка первого файла
Вторая строка первого файла
Третья строка первого файла
===== part2.txt =====
Первая строка второго файла
Вторая строка второго файла
Третья строка второго файла
Четвёртая строка второго файла
===== part3.txt =====
Первая строка третьего файла
Вторая строка третьего файла
Третья строка третьего файла

===== ОГЛАВЛЕНИЕ =====
part1.txt — начиная со строки 2
part2.txt — начиная со строки 6
part3.txt — начиная со строки 11

Общее количество перенесённых строк: 10
Размер итогового файла: 798 байт


# Задание 3. Сравнение содержимого двух файлов

In [4]:
def compare_files(name1, name2, report_name="diff_report.txt"):

    def normalize(line):
        """Применяет правила сравнения к строке."""
        line = line.rstrip("\n")          # убираем перевод строки
        return line

    differences = 0
    report_lines = []

    # Оба файла открываются в ОДНОМ операторе with
    with open(name1, "r", encoding="utf-8") as f1, \
         open(name2, "r", encoding="utf-8") as f2:

        line_no = 0
        while True:
            raw1 = f1.readline()
            raw2 = f2.readline()
            line_no += 1

            # Оба файла закончились
            if raw1 == "" and raw2 == "":
                break

            # Строки для сравнения (с учётом флагов)
            norm1 = normalize(raw1) if raw1 != "" else None
            norm2 = normalize(raw2) if raw2 != "" else None

            if norm1 == norm2:
                continue  # строки совпадают — идём дальше

            # -------- Строки различаются --------
            differences += 1
            report_lines.append(f"Строка {line_no} различается:")

            if norm1 is None:
                # В первом файле строки уже нет
                report_lines.append(f"   файл 1: <нет строки>")
                report_lines.append(f"   файл 2: {norm2}")
            elif norm2 is None:
                # Во втором файле строки уже нет
                report_lines.append(f"   файл 1: {norm1}")
                report_lines.append(f"   файл 2: <нет строки>")
            else:
                report_lines.append(f"   файл 1: {norm1}")
                report_lines.append(f"   файл 2: {norm2}")

    # -------- Формируем итоговый отчёт --------
    if differences == 0:
        report_lines.append("Файлы идентичны.")
    else:
        report_lines.append(f"Всего различий: {differences}")

    report_text = "\n".join(report_lines) + "\n"

    # Сохраняем отчёт в файл
    with open(report_name, "w", encoding="utf-8") as rep:
        rep.write(report_text)

    # Дублируем отчёт на экран
    print(report_text, end="")
    print(f"Отчёт сохранён в {report_name}")

    return differences


# ================== Демонстрация работы ==================
def _create_demo_files():
    """Создаёт два файла, слегка отличающихся друг от друга."""
    file1 = [
        "Начало работы",
        "Загрузка модулей",
        "Проверка соединения",
        "WARN  Использованы настройки по умолчанию",
        "Обработка данных",
        "Программа завершена",
        "Конец журнала",
    ]
    file2 = [
        "Начало работы",
        "Загрузка модулей",
        "Проверка соединения",
        "INFO  Использованы настройки по умолчанию",
        "Обработка данных",
        "Программа завершена",
    ]

    with open("file1.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(file1) + "\n")
    with open("file2.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(file2) + "\n")


if __name__ == "__main__":
    _create_demo_files()

    print("=== Сравнение без дополнительных флагов ===")
    count = compare_files("file1.txt", "file2.txt")

=== Сравнение без дополнительных флагов ===
Строка 4 различается:
   файл 1: WARN  Использованы настройки по умолчанию
   файл 2: INFO  Использованы настройки по умолчанию
Строка 7 различается:
   файл 1: Конец журнала
   файл 2: <нет строки>
Всего различий: 2
Отчёт сохранён в diff_report.txt


# Задание 4. Собственный контекстный менеджер


In [5]:
import os
import time
from contextlib import contextmanager
from datetime import datetime


# ============================================================
# Менеджер 1 — LogWriter (реализация через класс)
# ============================================================
class LogWriter:
    """
    Контекстный менеджер для ведения журнала.
    При входе открывает файл на дозапись и пишет строку о начале сеанса.
    При выходе пишет строку о завершении сеанса и закрывает файл.
    __enter__ возвращает объект с методом write(message).
    """

    def __init__(self, filename):
        self.filename = filename
        self.file = None
        self.start_time = None
        self.messages = 0          # счётчик записанных сообщений

    def __enter__(self):
        self.file = open(self.filename, "a", encoding="utf-8")
        self.start_time = time.time()
        stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.file.write(f"=== Сеанс начат: {stamp} ===\n")
        self.file.flush()
        return self                # возвращаем сам объект

    def write(self, message):
        """Дописывает сообщение с отметкой времени."""
        stamp = datetime.now().strftime("%H:%M:%S")
        self.file.write(f"{stamp} | {message}\n")
        self.file.flush()
        self.messages += 1

    def __exit__(self, exc_type, exc_val, exc_tb):
        duration = time.time() - self.start_time
        stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.file.write(
            f"=== Сеанс завершён, длительность {duration:.2f} с, "
            f"сообщений: {self.messages} ===\n"
        )
        self.file.close()
        # False => исключения НЕ подавляются
        return False


# ============================================================
# Менеджер 2 — safe_write (реализация через @contextmanager)
# ============================================================
@contextmanager
def safe_write(filename):
    """
    Атомарная запись в файл.
    Пишем во временный файл filename + '.tmp'.
    При успехе — переименовываем в основной (os.replace).
    При исключении — удаляем временный, основной файл не трогаем.
    """
    tmp_name = filename + ".tmp"
    f = open(tmp_name, "w", encoding="utf-8")
    try:
        yield f
    except Exception:
        f.close()
        if os.path.exists(tmp_name):
            os.remove(tmp_name)
        raise                       # обязательно пробрасываем исключение дальше
    else:
        f.close()
        os.replace(tmp_name, filename)   # атомарная замена
    # finally не нужен отдельно, но по требованиям можно и так:
    # (try/except/else выше покрывает все случаи)


# ============================================================
# Демонстрация
# ============================================================
def demo_log_writer():
    print("========== Демонстрация LogWriter ==========")

    # --- Сеанс 1 ---
    with LogWriter("journal.log") as log:
        log.write("Загрузка данных")
        log.write("Обработка завершена")
        log.write("Сохранение результата")

    time.sleep(1)  # небольшая пауза, чтобы длительность отличалась

    # --- Сеанс 2 (записи не должны затирать первый сеанс) ---
    with LogWriter("journal.log") as log:
        log.write("Повторный запуск")
        log.write("Проверка целостности")

    print("Содержимое journal.log:")
    print("-" * 50)
    with open("journal.log", "r", encoding="utf-8") as f:
        print(f.read(), end="")
    print("-" * 50)


def demo_safe_write():
    print("\n========== Демонстрация safe_write ==========")

    # Готовим исходный файл с "важными" данными
    with open("data.txt", "w", encoding="utf-8") as f:
        f.write("ИСХОДНОЕ СОДЕРЖИМОЕ\n")
        f.write("Очень важные данные\n")

    # --- Сценарий 1: успешная запись ---
    print("\n--- Сценарий 1: успешная запись ---")
    with safe_write("data.txt") as f:
        f.write("НОВОЕ СОДЕРЖИМОЕ\n")
        f.write("Записано успешно\n")

    print("Содержимое data.txt после успешной записи:")
    with open("data.txt", "r", encoding="utf-8") as f:
        print(f.read(), end="")

    # --- Сценарий 2: запись прервана исключением ---
    print("\n--- Сценарий 2: запись прервана исключением ---")
    try:
        with safe_write("data.txt") as f:
            f.write("ЭТО НЕ ДОЛЖНО ПОПАСТЬ В ФАЙЛ\n")
            raise RuntimeError("Сбой во время записи!")
    except RuntimeError as e:
        print(f"Перехвачено исключение: {e}")

    print("Содержимое data.txt после сбоя (должно остаться прежним):")
    with open("data.txt", "r", encoding="utf-8") as f:
        print(f.read(), end="")

    # Убедимся, что временный файл удалён
    print("Временный файл существует:",
          os.path.exists("data.txt.tmp"))


if __name__ == "__main__":
    demo_log_writer()
    demo_safe_write()

========== Демонстрация LogWriter ==========
Содержимое journal.log:
--------------------------------------------------
=== Сеанс начат: 2026-09-19 13:25:55 ===
13:25:55 | Загрузка данных
13:25:55 | Обработка завершена
13:25:55 | Сохранение результата
=== Сеанс завершён, длительность 0.00 с, сообщений: 3 ===
=== Сеанс начат: 2026-09-19 13:25:56 ===
13:25:56 | Повторный запуск
13:25:56 | Проверка целостности
=== Сеанс завершён, длительность 0.00 с, сообщений: 2 ===
--------------------------------------------------

========== Демонстрация safe_write ==========

--- Сценарий 1: успешная запись ---
Содержимое data.txt после успешной записи:
НОВОЕ СОДЕРЖИМОЕ
Записано успешно

--- Сценарий 2: запись прервана исключением ---
Перехвачено исключение: Сбой во время записи!
Содержимое data.txt после сбоя (должно остаться прежним):
НОВОЕ СОДЕРЖИМОЕ
Записано успешно
Временный файл существует: False
